In [1]:
print("Hello, world!")

Hello, world!


### (a) What Unicode character does chr(0) return?

In [14]:
for i in range(10):
    print(repr(chr(i)))

'\x00'
'\x01'
'\x02'
'\x03'
'\x04'
'\x05'
'\x06'
'\x07'
'\x08'
'\t'


It's a control char. In C-like languages it was needed to mark that this is the end of string and functions as `strlen` could understand that explicitly.

### (b) How does this character’s string representation (__repr__()) differ from its printed representation?

In [21]:
name = "Andrew\nHello, world!\t"
print(name, repr(name), sep="\n")

Andrew
Hello, world!	
'Andrew\nHello, world!\t'


`rerp()` function doesn't convert special unicodes bytes into readable string. As in the example above, the symbol `\n`, that representes the newline, has no sense in a raw representation (for ordinary people)

### (c) What happens when this character occurs in text? It may be helpful to play around with the following in your Python interpreter and see if it matches your expectations:
```python
>>> chr(0)
>>> print(chr(0))
>>> "this is a test" + chr(0) + "string"
>>> print("this is a test" + chr(0) + "string")
```

In [24]:
chr(0)

'\x00'

In [26]:
print(chr(0))

 


In [27]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [28]:
print("this is a test" + chr(0) + "string")

this is a test string


In [29]:
len("this is a teststring"), len("this is a test" + chr(0) + "string")

(20, 21)

In [30]:
null_string = "this is a test" + chr(0) + "string"
just_string = "this is a teststring"

null_string.encode(), just_string.encode()

(b'this is a test\x00string', b'this is a teststring')

In [31]:
null_string.encode().decode(), just_string.encode().decode()

('this is a test\x00string', 'this is a teststring')

When this character occurs in text, nothing happens, except that the length of the input becomes much higher and the next calculcation with a tokenizer becomes less effecient

### (a) What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than UTF-16 or UTF-32? It may be helpful to compare the output of these encodings for various input strings.

In [3]:
# Let's take a string for the 1-st lecture

string = "Hello, 🌍! 你好!"

# And then we'll decode it, using 3 algorithms
utf_8 = string.encode("utf-8")
utf_16 = string.encode("utf-16")
utf_32 = string.encode("utf-32")
print(len(utf_8), len(utf_16), len(utf_32), end="\n\n")
utf_8, utf_16, utf_32

20 30 56



(b'Hello, \xf0\x9f\x8c\x8d! \xe4\xbd\xa0\xe5\xa5\xbd!',
 b'\xff\xfeH\x00e\x00l\x00l\x00o\x00,\x00 \x00<\xd8\r\xdf!\x00 \x00`O}Y!\x00',
 b'\xff\xfe\x00\x00H\x00\x00\x00e\x00\x00\x00l\x00\x00\x00l\x00\x00\x00o\x00\x00\x00,\x00\x00\x00 \x00\x00\x00\r\xf3\x01\x00!\x00\x00\x00 \x00\x00\x00`O\x00\x00}Y\x00\x00!\x00\x00\x00')

In [8]:
print("========UTF-8========")
for i in utf_8:
    print(i, end=" ")
print("\n========UTF-16========")
for i in utf_16:
    print(i, end=" ")
print("\n========UTF-32========")
for i in utf_32:
    print(i, end=" ")

========UTF-8========
72 101 108 108 111 44 32 240 159 140 141 33 32 228 189 160 229 165 189 33 
========UTF-16========
255 254 72 0 101 0 108 0 108 0 111 0 44 0 32 0 60 216 13 223 33 0 32 0 96 79 125 89 33 0 
========UTF-32========
255 254 0 0 72 0 0 0 101 0 0 0 108 0 0 0 108 0 0 0 111 0 0 0 44 0 0 0 32 0 0 0 13 243 1 0 33 0 0 0 32 0 0 0 96 79 0 0 125 89 0 0 33 0 0 0 

So, we can see `utf-8` is the most effecient over the all utf-like encoders, because it produces less bytes (there is no `control chars`, `unusual symbols` is converted into more compact form)
Also the problem is that `utf-16` and `utf-32` declare more bytes to represent char, even if there is no need to use 4 bytes

### (b) Consider the following (incorrect) function, which is intended to decode a UTF-8 byte string into a Unicode string. Why is this function incorrect? Provide an example of an input byte string that yields incorrect results.
```python
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

>>> decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))
'hello'

In [9]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

# So, it really works with one-byte symbols
decode_utf8_bytes_to_str_wrong("hello, my name is Andrew".encode("utf-8"))

'hello, my name is Andrew'

In [ ]:
# Buf if we take the previous example, it won't be good
decode_utf8_bytes_to_str_wrong(string.encode("utf-8"))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf0 in position 0: unexpected end of data

It happens, because in the encoding every symbol has dynamic representation length hence globus emoji couldn't be converted into char representation as it has more than 1 byte

We should revise this function as it should encode a whole array of bytes:

In [11]:
def decode_utf8_bytes_to_str_correct(bytestring: bytes):
    return bytestring.decode()

decode_utf8_bytes_to_str_correct(string.encode("utf-8"))

'Hello, 🌍! 你好!'

### (c) Give a two-byte sequence that does not decode to any Unicode character(s).

In [12]:
b'\xc0\x80'.decode('utf-8')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc0 in position 0: invalid start byte